# Balanced Pool

The `balanced_pool()` function returns 42 pre-configured generators covering 15 distinct behavioral niches. This avoids implicit bias toward any single domain (e.g., finance) by allocating generators proportionally to each behavioral category's complexity range.

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from synforecast import SynSet, balanced_pool

## Generate a Balanced Dataset

Create 42 generators and generate one series per generator.

In [ ]:
generators = balanced_pool(
    min_length=200, max_length=200, freq="D", seed=42, engine="polars"
)
dataset = SynSet(generators)
df = dataset.generate(n_series_per_generator=1)

print(f"Generators: {len(generators)}")
print(f"Series: {df['unique_id'].n_unique()}")
print(f"Total observations: {len(df)}")

## Generator Names

Each generator has a descriptive name indicating its type and configuration.

In [ ]:
for i, gen in enumerate(generators):
    print(f"  {i}: {gen.alias}")

## Overview: All 42 Series

A compact grid showing every series in the balanced pool.

In [ ]:
fig, axes = plt.subplots(7, 6, figsize=(18, 16))
axes = axes.flatten()

for i, gen in enumerate(generators):
    uid = str(i)
    series = df.filter(pl.col("unique_id") == uid)
    values = series["y"].to_list()
    axes[i].plot(values, linewidth=0.8)
    axes[i].set_title(gen.alias, fontsize=7)
    axes[i].tick_params(labelsize=5)

fig.suptitle("Balanced Pool: 42 Generators across 15 Behavioral Niches", fontsize=14)
plt.tight_layout()
plt.show()

## Niche Deep-Dives

Each behavioral niche contributes a different number of generators. Below we group them by niche and plot the variants side by side.

In [ ]:
niches = [
    ("ARMA + Seasonality (SARIMA)", list(range(0, 5))),
    ("Exponential Smoothing (ETS)", list(range(5, 9))),
    ("Long-Range Memory (FBM)", list(range(9, 12))),
    ("Structural Breaks (Regime Switching)", list(range(12, 14))),
    ("Volatility Clustering (GARCH)", list(range(14, 16))),
    ("Irregular Cycles (Cyclic)", list(range(16, 18))),
    ("Sparse/Intermittent (Intermittent Demand)", list(range(18, 21))),
    ("Multi-Seasonal (Energy Load)", list(range(21, 23))),
    ("Sensor Artifacts (IoT Sensor)", list(range(23, 26))),
    ("Physiological (Vital Signs)", list(range(26, 29))),
    ("Smooth/Rough Functions (Gaussian Process)", list(range(29, 33))),
    ("Deterministic Chaos (Chaotic System)", list(range(33, 36))),
    ("Count Time Series (INAR)", list(range(36, 38))),
    ("Bounded/Proportion Data (Bounded Process)", list(range(38, 40))),
    ("Heavy-Tailed Processes (Levy Process)", list(range(40, 42))),
]

In [ ]:
for niche_name, indices in niches:
    n = len(indices)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), squeeze=False)
    fig.suptitle(niche_name, fontsize=12, fontweight="bold")

    for j, idx in enumerate(indices):
        uid = str(idx)
        series = df.filter(pl.col("unique_id") == uid)
        axes[0][j].plot(series["y"].to_list(), linewidth=0.9)
        axes[0][j].set_title(generators[idx].alias, fontsize=8)
        axes[0][j].tick_params(labelsize=7)

    plt.tight_layout()
    plt.show()

## Summary Statistics

Compare key statistics across all 42 series to see how the balanced pool spans different value ranges and variabilities.

In [ ]:
stats = (
    df.group_by("unique_id")
    .agg(
        [
            pl.col("y").count().alias("count"),
            pl.col("y").min().alias("min"),
            pl.col("y").max().alias("max"),
            pl.col("y").mean().alias("mean"),
            pl.col("y").std().alias("std"),
        ]
    )
    .sort("unique_id")
)
stats

## Scaling Up

Generate multiple series per generator for a larger dataset.

In [ ]:
df_large = dataset.generate(n_series_per_generator=5)
print(f"Series: {df_large['unique_id'].n_unique()}")
print(f"Total observations: {len(df_large)}")

Plot five series from a single generator to see intra-generator variation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Series 0-4 come from the first generator (Stationary AR(1))
for sid in range(5):
    uid = str(sid)
    series = df_large.filter(pl.col("unique_id") == uid)
    ax.plot(series["y"].to_list(), alpha=0.7, label=uid)

ax.set_title(f"Intra-Generator Variation: {generators[0].alias}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()